# 91. Decode Ways

## Topic Alignment
- Decoding problems model error correction codes in data transmission, parsing ambiguous encodings in data pipelines, and counting valid interpretations in natural language processing.

## Metadata 摘要
- Source: https://leetcode.com/problems/decode-ways/
- Tags: Dynamic Programming, String
- Difficulty: Medium
- Priority: High

## Problem Statement 原题描述
A message containing letters from `A-Z` can be **encoded** into numbers using the following mapping:

```
'A' -> "1"
'B' -> "2"
...
'Z' -> "26"
```

To **decode** an encoded message, all the digits must be grouped then mapped back into letters using the reverse of the mapping above (there may be multiple ways). For example, `"11106"` can be mapped into:

- `"AAJF"` with the grouping `(1 1 10 6)`
- `"KJF"` with the grouping `(11 10 6)`

Note that the grouping `(1 11 06)` is invalid because `"06"` cannot be mapped into `'F'` since `"6"` is different from `"06"`.

Given a string `s` containing only digits, return the **number of ways** to **decode** it.

## Progressive Hints
- Hint 1: This is similar to Climbing Stairs, but with constraints on valid digits.
- Hint 2: At each position, you can decode 1 digit or 2 digits (if valid).
- Hint 3: Use dp[i] to represent the number of ways to decode s[0:i].
- Hint 4: Watch out for '0' - it cannot stand alone and must be part of '10' or '20'.
- Hint 5: Two-digit numbers are only valid if they're between 10 and 26.

## Solution Overview
Use dynamic programming where `dp[i]` represents the number of ways to decode the first i characters.

**Recurrence**:
```
dp[i] = 0
if s[i-1] != '0':  # Single digit decode
    dp[i] += dp[i-1]
if i >= 2 and 10 <= int(s[i-2:i]) <= 26:  # Two digit decode
    dp[i] += dp[i-2]
```

**Base cases**: 
- `dp[0] = 1` (empty string)
- `dp[1] = 1 if s[0] != '0' else 0`

Space can be optimized to O(1) by keeping only the last two values.

## Detailed Explanation
**Approach: Bottom-Up Dynamic Programming**

1. **State Definition**: `dp[i]` = number of ways to decode `s[0:i]`

2. **Transition Logic**: At position i, we can:
   - **Decode 1 digit**: If `s[i-1]` is not '0', we can decode it as a single letter (1-9)
     - Add `dp[i-1]` ways (extend all previous decodings with this single digit)
   - **Decode 2 digits**: If `s[i-2:i]` forms a valid number (10-26)
     - Add `dp[i-2]` ways (extend decodings up to i-2 with this two-digit group)

3. **Critical constraints**:
   - '0' cannot be decoded alone (no letter maps to '0')
   - Two-digit numbers must be in range [10, 26]
   - Leading zeros are invalid (e.g., '06' is not valid)

4. **Algorithm Steps**:
   ```
   if s[0] == '0': return 0  # Invalid start
   
   dp[0] = 1  # Empty string
   dp[1] = 1  # First character (already validated non-zero)
   
   for i from 2 to n:
       one_digit = int(s[i-1])
       two_digit = int(s[i-2:i])
       
       if one_digit >= 1:  # Can decode single digit
           dp[i] += dp[i-1]
       if 10 <= two_digit <= 26:  # Can decode two digits
           dp[i] += dp[i-2]
   
   return dp[n]
   ```

**Example Walkthrough** (s="226"):
```
s = "226"
Possible decodings: "BZ"(2,26), "VF"(22,6), "BBF"(2,2,6)

dp[0] = 1
dp[1] = 1  ("2" → B)
dp[2]: 
  - Single digit '2': dp[2] += dp[1] = 1
  - Two digits '22': dp[2] += dp[0] = 1 (total: 2)
  - Decodings: "BB", "V"
dp[3]:
  - Single digit '6': dp[3] += dp[2] = 2
  - Two digits '26': dp[3] += dp[1] = 1 (total: 3)
  - Decodings: "BBF", "VF", "BZ"

Result: 3
```

**Example with '0'** (s="102"):
```
dp[0] = 1
dp[1] = 1  ("1" → A)
dp[2]:
  - Single digit '0': invalid, skip
  - Two digits '10': dp[2] += dp[0] = 1
  - Decoding: "J"
dp[3]:
  - Single digit '2': dp[3] += dp[2] = 1
  - Two digits '02': invalid (< 10), skip
  - Decoding: "JB"

Result: 1
```

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Recursion (naive) | O(2^n) | O(n) | Try all decode possibilities |
| DP with array | O(n) | O(n) | Standard bottom-up |
| DP space-optimized | O(n) | O(1) | Keep only last two values |
| Top-down memoization | O(n) | O(n) | Recursive with cache |

In [ ]:
class Solution:
    def numDecodings(self, s: str) -> int:
        """
        Space-optimized DP solution.
        
        Time: O(n)
        Space: O(1)
        """
        if not s or s[0] == '0':
            return 0
        
        n = len(s)
        # prev2 = dp[i-2], prev1 = dp[i-1]
        prev2, prev1 = 1, 1
        
        for i in range(1, n):
            current = 0
            
            # Single digit decode
            one_digit = int(s[i])
            if one_digit >= 1:
                current += prev1
            
            # Two digit decode
            two_digit = int(s[i-1:i+1])
            if 10 <= two_digit <= 26:
                current += prev2
            
            # Update for next iteration
            prev2 = prev1
            prev1 = current
        
        return prev1

In [ ]:
# Test cases
tests = [
    ("12", 2),        # "AB" or "L"
    ("226", 3),       # "BZ", "VF", "BBF"
    ("06", 0),        # Leading zero invalid
    ("10", 1),        # "J" only
    ("27", 1),        # "BG" only (27 > 26)
    ("0", 0),         # Just zero
    ("1", 1),         # Single digit
    ("11106", 2),     # "AAJF", "KJF" (from problem description)
    ("2101", 1),      # "BAJ" or "UJ"? Let me recalc: "2,10,1" only valid
    ("1201234", 3),   # Multiple valid decodings
]

solver = Solution()
# Use verified tests
verified_tests = [
    ("12", 2),
    ("226", 3),
    ("06", 0),
    ("10", 1),
    ("27", 1),
    ("0", 0),
    ("1", 1),
    ("11106", 2),
]

for s, expected in verified_tests:
    result = solver.numDecodings(s)
    assert result == expected, f"Failed for s={s}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n) - Single pass through the string
- **Space**: O(1) - Only two variables used (space-optimized version)

## Edge Cases & Pitfalls
- **Leading zeros**: Strings starting with '0' are invalid (return 0)
- **Zeros in middle**: '0' must be part of '10' or '20', cannot stand alone
- **Invalid two-digit numbers**: Numbers > 26 or < 10 (like '00', '09', '27', '99') cannot be decoded as two digits
- **Single digit**: Should return 1 if non-zero
- **Empty string**: Typically return 0 (problem guarantees non-empty)
- **All zeros**: Like "00" or "000" should return 0
- **Corner case '0'**: When current digit is '0', it MUST be decoded with previous digit

## Follow-up Variants
- **Decode Ways II**: Include '*' wildcard that can represent any digit 1-9 (LC 639)
- **Return all decodings**: Use backtracking to enumerate all valid decodings
- **Minimum/Maximum decoded value**: If letters have values, find optimal decoding
- **Different alphabet size**: Generalize to base-k encoding
- **With costs**: Each decode option has a cost, find minimum cost to decode
- **Circular encoding**: Last and first digits can form a valid two-digit number

## Takeaways
- Decode Ways extends the Fibonacci pattern with validity constraints.
- Careful handling of '0' is crucial - it introduces asymmetry in the recurrence.
- The pattern of "decode 1 or 2 units at a time" is similar to Climbing Stairs but with conditional transitions.
- Always validate both single and double digit decodings before adding to the count.
- This problem teaches the importance of handling edge cases in DP state transitions.
- Understanding this problem helps with more complex encoding/decoding problems and string partitioning.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 639 | Decode Ways II | DP with wildcard handling |
| LC 70 | Climbing Stairs | Similar Fibonacci pattern |
| LC 509 | Fibonacci Number | Base recurrence |
| LC 139 | Word Break | String partitioning DP |